# Daily Challenge: Minimal MCP Weather server & client (no LLM)

**Goal:** a minimal **MCP server** (`WeatherDemo`) and **client** that talk over **STDIO**. The server exposes:
- one **tool**, `get_weather(city)`, an *action* that takes arguments and returns a dict;
- one **resource**, `cities://list`, *read-only context* identified by a URI.

The client spawns the server, discovers its capabilities and invokes them. No model is involved: this is exactly the plumbing an LLM host (Claude Desktop, an IDE agent…) would use.

| | Role here |
|---|---|
| **Server** | `server.py`: owns the data and the capabilities and answers JSON-RPC requests |
| **Client** | `client.py`: spawns the server, `initialize`, `resources/list`, `tools/list`, `resources/read`, `tools/call` |
| **Transport** | STDIO: JSON-RPC messages on the subprocess's **stdin/stdout**; logs must go to **stderr** |
| **Tool vs resource** | A tool *does* something with arguments (model-controlled); a resource *provides* data by URI (application-controlled) |

## Setup

In [1]:
# Course command: pip install "mcp[cli]"
# mcp 2.x renamed FastMCP (-> MCPServer) and breaks `from mcp.server.fastmcp import FastMCP`,
# so we pin the v1 SDK that the exercise is written for.
%pip install -q "mcp[cli]<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
!python --version
!python -c "from importlib.metadata import version; print('mcp', version('mcp'))"
!mcp --help | head -14

Python 3.13.16


mcp 1.30.0


                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ version  Show the MCP version.                                               │
│ dev      Run an MCP server with the MCP Inspector.                           │
│ run      Run an MCP server.                                                  │
│ install  Install an MCP se

## 1. Server (`server.py`)
- `FastMCP("WeatherDemo")` creates the server.
- `get_weather(city: str) -> dict` does a case-insensitive lookup in a small in-memory dict (Paris, London, NYC, Tel Aviv). An unknown city returns an **error dict** (with the supported cities) instead of raising, so the client always gets a structured answer.
- `cities://list` returns the supported cities, one per line.
- `logging` goes to **stderr**: on the STDIO transport, anything printed to stdout would corrupt the JSON-RPC stream.
- `mcp.run()` starts the STDIO loop. With `mcp run server.py`, the CLI imports the module and starts the `mcp` object itself, so the `__main__` block only runs with `python server.py`.

In [3]:
%%writefile server.py
# server.py - minimal MCP weather server (no LLM): one tool + one resource over STDIO
import logging
import sys

from mcp.server.fastmcp import FastMCP

# Log to stderr: on the STDIO transport, stdout is reserved for JSON-RPC messages.
logging.basicConfig(level=logging.INFO, stream=sys.stderr,
                    format="[server] %(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("weather")

mcp = FastMCP("WeatherDemo")

# Small static, in-memory "database"
WEATHER = {
    "Paris": {"temp_c": 21, "condition": "sunny", "humidity_pct": 48},
    "London": {"temp_c": 15, "condition": "cloudy", "humidity_pct": 72},
    "NYC": {"temp_c": 26, "condition": "humid", "humidity_pct": 80},
    "Tel Aviv": {"temp_c": 30, "condition": "clear", "humidity_pct": 60},
}


@mcp.tool()
def get_weather(city: str) -> dict:
    """Return static weather data for a supported city (see the cities://list resource)."""
    log.info("get_weather called with city=%r", city)
    match = next((c for c in WEATHER if c.lower() == city.strip().lower()), None)
    if match is None:
        log.warning("unknown city %r", city)
        return {"error": f"City '{city}' not supported", "supported": list(WEATHER)}
    return {"city": match, **WEATHER[match]}


@mcp.resource("cities://list")
def list_cities() -> str:
    """Newline-separated list of the supported cities."""
    log.info("cities://list read")
    return "\n".join(WEATHER)


if __name__ == "__main__":
    log.info("WeatherDemo server starting on STDIO")
    mcp.run()  # default transport: stdio

Overwriting server.py


## 2. Client (`client.py`)
It spawns the server with `StdioServerParameters(command="mcp", args=["run", "server.py"])`, then `initialize()`, lists the resources and tools, reads `cities://list` and calls `get_weather` for a valid city (Paris) and an unsupported one (Berlin). The tool's dict comes back as **structured content**, with a JSON text copy for older clients.

In [4]:
%%writefile client.py
# client.py - spawns the WeatherDemo server over STDIO, discovers and invokes its features
import asyncio
import json

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            print(f"Connected to: {init.serverInfo.name}")

            resources = await session.list_resources()
            print("Resources:", [str(r.uri) for r in resources.resources])

            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])
            for t in tools.tools:
                print(f"  {t.name}: {t.description}  args={list(t.inputSchema['properties'])}")

            cities = await session.read_resource("cities://list")
            print("cities://list ->")
            print(cities.contents[0].text)

            for city in ["Paris", "Berlin"]:  # a valid city and an unsupported one
                result = await session.call_tool("get_weather", arguments={"city": city})
                data = result.structuredContent or json.loads(result.content[0].text)
                print(f"get_weather({city!r}) ->", data)


if __name__ == "__main__":
    asyncio.run(run())

Overwriting client.py


## 3. Run and observe
### Single terminal: `python client.py` (the client spawns the server)
With `2>&1`, the server's stderr logs (`[server] …`) are interleaved with the client's prints, so we can **see each request arrive** at the server.

In [5]:
!python client.py 2>&1 | tee terminal_capture.txt

Connected to: WeatherDemo
[server] 2026-10-12 12:50:44,665 INFO Processing request of type ListResourcesRequest
Resources: ['cities://list']
[server] 2026-10-12 12:50:44,668 INFO Processing request of type ListToolsRequest
Tools: ['get_weather']
  get_weather: Return static weather data for a supported city (see the cities://list resource).  args=['city']


[server] 2026-10-12 12:50:44,670 INFO Processing request of type ReadResourceRequest
[server] 2026-10-12 12:50:44,671 INFO cities://list read
cities://list ->
Paris
London
NYC
Tel Aviv
[server] 2026-10-12 12:50:44,673 INFO Processing request of type CallToolRequest
[server] 2026-10-12 12:50:44,673 INFO get_weather called with city='Paris'
get_weather('Paris') -> {'city': 'Paris', 'temp_c': 21, 'condition': 'sunny', 'humidity_pct': 48}
[server] 2026-10-12 12:50:44,675 INFO Processing request of type CallToolRequest
[server] 2026-10-12 12:50:44,675 INFO get_weather called with city='Berlin'
[server] 2026-10-12 12:50:44,675 WARNING unknown city 'Berlin'
get_weather('Berlin') -> {'error': "City 'Berlin' not supported", 'supported': ['Paris', 'London', 'NYC', 'Tel Aviv']}


In [6]:
# Client output only (stdout)
!python client.py 2>/dev/null

Connected to: WeatherDemo
Resources: ['cities://list']
Tools: ['get_weather']
  get_weather: Return static weather data for a supported city (see the cities://list resource).  args=['city']
cities://list ->
Paris
London
NYC
Tel Aviv
get_weather('Paris') -> {'city': 'Paris', 'temp_c': 21, 'condition': 'sunny', 'humidity_pct': 48}
get_weather('Berlin') -> {'error': "City 'Berlin' not supported", 'supported': ['Paris', 'London', 'NYC', 'Tel Aviv']}


### Two terminals (debugging): `mcp run server.py`
Started alone, the server waits for a client on stdin. Here we check for a few seconds that it boots without errors.

In [7]:
import subprocess
p = subprocess.Popen(["mcp", "run", "server.py"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
try:
    p.wait(timeout=5)                      # keep stdin open (closing it = EOF = normal shutdown)
    print("Server exited early! stderr:\n", p.stderr.read())
except subprocess.TimeoutExpired:
    print("✅ `mcp run server.py` is up and waiting for an MCP client on STDIO.")
    p.kill()

✅ `mcp run server.py` is up and waiting for an MCP client on STDIO.


### Seeing the data flow: the raw JSON-RPC messages
The SDK hides the wire format. Here we act as a client by hand and print every request and response that crosses the STDIO pipe.

In [8]:
import json
msgs = [
    {"jsonrpc": "2.0", "id": 1, "method": "initialize",
     "params": {"protocolVersion": "2024-11-05", "capabilities": {}, "clientInfo": {"name": "raw-client", "version": "0.1"}}},
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    {"jsonrpc": "2.0", "id": 2, "method": "resources/list"},
    {"jsonrpc": "2.0", "id": 3, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 4, "method": "resources/read", "params": {"uri": "cities://list"}},
    {"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "get_weather", "arguments": {"city": "London"}}},
]
p = subprocess.Popen(["python", "server.py"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
for m in msgs:
    p.stdin.write(json.dumps(m) + "\n"); p.stdin.flush()
    print("CLIENT -> SERVER:", json.dumps(m)[:160])
    if "id" in m:
        reply = json.loads(p.stdout.readline())
        print("SERVER -> CLIENT:", json.dumps(reply)[:400], "\n")
p.kill()

CLIENT -> SERVER: {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2024-11-05", "capabilities": {}, "clientInfo": {"name": "raw-client", "versio


SERVER -> CLIENT: {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": "2024-11-05", "capabilities": {"experimental": {}, "prompts": {"listChanged": false}, "resources": {"subscribe": false, "listChanged": false}, "tools": {"listChanged": false}}, "serverInfo": {"name": "WeatherDemo", "version": "1.30.0"}}} 

CLIENT -> SERVER: {"jsonrpc": "2.0", "method": "notifications/initialized"}
CLIENT -> SERVER: {"jsonrpc": "2.0", "id": 2, "method": "resources/list"}
SERVER -> CLIENT: {"jsonrpc": "2.0", "id": 2, "result": {"resources": [{"name": "list_cities", "uri": "cities://list", "description": "Newline-separated list of the supported cities.", "mimeType": "text/plain"}]}} 

CLIENT -> SERVER: {"jsonrpc": "2.0", "id": 3, "method": "tools/list"}
SERVER -> CLIENT: {"jsonrpc": "2.0", "id": 3, "result": {"tools": [{"name": "get_weather", "description": "Return static weather data for a supported city (see the cities://list resource).", "inputSchema": {"properties": {"city": {"title": "City", 

## What to submit
- `server.py` and `client.py` (written above and saved next to this notebook in the repository);
- the terminal capture `terminal_capture.txt`, showing:
  - **resource list:** `['cities://list']`;
  - **tool list:** `['get_weather']`;
  - **`cities://list`:** Paris / London / NYC / Tel Aviv;
  - **`get_weather('Paris')`:** `{'city': 'Paris', 'temp_c': 21, 'condition': 'sunny', 'humidity_pct': 48}`, plus the error dict for an unsupported city.

### Troubleshooting notes
- `Connection closed` → run `mcp run server.py` alone to see the server error. The most likely cause today is having **mcp 2.x** installed with this v1 code: pin `mcp[cli]<2`.
- **Never `print()` in an STDIO server**: it writes to stdout and breaks the protocol. Log to stderr, as done here.
- Tool argument types must match the signature (`{"city": "Paris"}`, a string).